# Opportunity Intelligence — Full Training Pipeline

**Runtime: L4 GPU (Runtime → Change runtime type → L4 GPU)**

Trains both capabilities on dataset v4 with Drive-durable checkpoints.
Safe to close the browser — Colab Pro background execution keeps the runtime alive.
All results sync to `MyDrive/opportunity_slm_runs/` continuously.

**Estimated time: ~2 hours total** (classifier ~45 min + summarizer ~60 min + setup)

In [ ]:
# ===== CELL 1: GPU + Drive setup =====
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DRIVE_SYNC_DIR'] = '/content/drive/MyDrive/opportunity_slm_runs'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ROOT = '/content/drive/MyDrive/opportunity_slm_runs'
os.makedirs(f'{ROOT}/datasets', exist_ok=True)
os.makedirs(f'{ROOT}/models', exist_ok=True)
os.makedirs(f'{ROOT}/manifests', exist_ok=True)
print('Drive synced root:', ROOT)
print('GPU ready')

In [ ]:
# ===== CELL 2: clone repo + install deps =====
!git clone -q https://github.com/sadatdaniel/opportunity-slm-training.git
%cd /content/opportunity-slm-training
!pip install -q uv
!uv sync --group training 2>&1 | tail -1
print('deps installed')

In [ ]:
# ===== CELL 3: extract datasets from Drive (or upload if missing) =====
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/opportunity_slm_runs'

for name in ['classifier_dataset_v4.tar.gz', 'summarizer_dataset_v4.tar.gz']:
    drive_path = f'{DRIVE_ROOT}/datasets/{name}'
    local_path = f'/content/{name}'
    if not os.path.isfile(local_path):
        if os.path.isfile(drive_path):
            shutil.copy2(drive_path, local_path)
            print(f'copied {name} from Drive')
        else:
            print(f'WARNING: {name} not found on Drive or /content — upload via Files pane')

!mkdir -p datasets/classifier datasets/summarizer
!tar xzf /content/classifier_dataset_v4.tar.gz -C datasets/classifier 2>/dev/null
!tar xzf /content/summarizer_dataset_v4.tar.gz -C datasets/summarizer 2>/dev/null
print('classifier v4:', os.path.isfile('datasets/classifier/v4/train.jsonl'))
print('summarizer v4:', os.path.isfile('datasets/summarizer/v4/train.jsonl'))

In [ ]:
# ===== CELL 4: train Qwen3-0.6B classifier (~40 min) =====
# DriveSyncCallback active — checkpoints sync to Drive per epoch
import subprocess
r = subprocess.run(
    ['uv', 'run', 'python', '-m', 'training.classifier.train',
     '--config', 'training/classifier/config_qwen06.yaml'],
    text=True
)
print(f'classifier train exit: {r.returncode}')
assert r.returncode == 0, 'training failed'

In [ ]:
# ===== CELL 5: evaluate classifier on frozen test split =====
!uv run python -m training.classifier.evaluate --model models/classifier/v0.3.0-qwen06 --dataset-version v4

# sync eval report to Drive
import glob, shutil
for f in glob.glob('reports/classifier_eval_*.json'):
    shutil.copy2(f, f'{DRIVE_ROOT}/manifests/')

In [ ]:
# ===== CELL 6: train Qwen3-4B LoRA summarizer (~60-90 min) =====
import subprocess
r = subprocess.run(
    ['uv', 'run', 'python', '-m', 'training.summarizer.train',
     '--config', 'training/summarizer/config_qwen4b_lora.yaml'],
    text=True
)
print(f'summarizer train exit: {r.returncode}')
assert r.returncode == 0, 'training failed'

In [ ]:
# ===== CELL 7: evaluate summarizer on frozen test split =====
!uv run python -m training.summarizer.evaluate --model models/summarizer/v0.3.0-qwen4b-lora --dataset-version v4

import glob, shutil
for f in glob.glob('reports/summarizer_eval_*.json'):
    shutil.copy2(f, f'{DRIVE_ROOT}/manifests/')

In [ ]:
# ===== CELL 8: sync ALL artifacts to Drive =====
import glob, shutil, os

for cap in ['classifier', 'summarizer']:
    model_dir = f'models/{cap}'
    if os.path.isdir(model_dir):
        for version in os.listdir(model_dir):
            dst = f'{DRIVE_ROOT}/models/{cap}/{version}'
            os.makedirs(dst, exist_ok=True)
            for f in glob.glob(f'{model_dir}/{version}/*'):
                shutil.copy2(f, dst + '/' + os.path.basename(f))

for f in glob.glob('experiments/manifests/*.yaml'):
    shutil.copy2(f, f'{DRIVE_ROOT}/manifests/')

print('ALL ARTIFACTS SYNCED TO DRIVE')
print('models:', os.listdir(f'{DRIVE_ROOT}/models'))
print('manifests:', os.listdir(f'{DRIVE_ROOT}/manifests'))

# DONE

Both models trained, evaluated, and safely on Drive.

## Results so far

| model | accuracy / metric | status |
|---|---|---|
| Supra classifier v0.2.0 | 84.3% (v2 test) | released |
| Qwen3-0.6B classifier v0.3.0 | from Cell 5 above | this run |
| Qwen3-4B LoRA summarizer | from Cell 7 above | this run |

## Files on Drive

- `MyDrive/opportunity_slm_runs/models/` — trained model artifacts
- `MyDrive/opportunity_slm_runs/manifests/` — run manifests + eval reports
- `MyDrive/opportunity_slm_runs/summarizer_*/checkpoint-*/` — mid-training checkpoints